# C06 Fake and unfounded request risk
Civic Operations · Nabeel Taresh · Group 23

## 1. Goal and contract

Score P(unfounded). Actions: PROCEED / LOWER_PRIORITY / HOLD_FOR_REVIEW.
Never REJECT. Failure: probability None, PROCEED.

Read `05_Team_Workspace/C05_C06_changelog/RESULTS.md`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "01_Application" / "data" / "sample").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "01_Application"))
import pandas as pd
pd.set_option("display.max_columns", 40)
print("repo:", ROOT)

## 2. Data

Label: `verified_outcome` in {GENUINE, UNFOUNDED}. Drop inconclusive.
**Never** use `resolution_description` as a feature. Reporter ids from the
shared synthetic links only. `hidden_honesty` is evaluation-only.

In [ ]:
from components.c06_fake import LEAKAGE_COLUMNS, unfounded_base_rates
print("leakage columns blocked:", sorted(LEAKAGE_COLUMNS))
print("type base rates (P unfounded) sample:", list(unfounded_base_rates().items())[:5])

## 3. Baseline

Rules: type base rate, burst at 5/hour, history, corroboration always lowers risk.

## 4. Proposed method

Production `hybrid_rules_logreg`: calibrated logistic regression plus the rule
layer. GBM is the nonlinear arm. Ablation: `logreg_no_burst`.

## 5. Evaluation table

`py 06_Workspace_Tools/c05_c06_eval.py benchmark` → `03_Research/experiments/c06_benchmark.csv`.
Metrics: AUC, Brier, F1 at 0.28, FPR on first-time reporters, recall on urgent
unfounded types, latency.

In [ ]:
p = ROOT / "experiments" / "c06_benchmark.csv"
if p.exists():
    display(pd.read_csv(p))
else:
    print("Run: py 06_Workspace_Tools/c05_c06_eval.py benchmark")

## 6. Error analysis

First-time FPR is the fairness check. Urgent recall is the safety check.
NYPD “no evidence” notes inflate UNFOUNDED. Treat AUC as a proxy score.

## 7. Decision

Production: `hybrid_rules_logreg` at VERSION 0.1.0. Pick by Brier and first-time
FPR, with a floor on urgent recall.

## 8. Artifact and version

`components/c06_fake.py` · VERSION 0.1.0 ·
`contributing_factors` reasons · changelog in `05_Team_Workspace/C05_C06_changelog/`.